# MvDEC aligned-balanced alignment-0.05 seed-42 gate: REUTERS, RCV1, 20NEWS

Runs the corrected `mvdec_2025_aligned_balanced_alignment_005_full_batch_v1` protocol for exactly seed 42 in the requested order: REUTERS, RCV1, then canonical 20NEWS last. This is a fusion-collapse correction diagnostic, not an exact-paper reproduction. The notebook never substitutes a smaller batch after OOM and never packages a partial matrix as success.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle/Colab GPU before running this notebook."


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

TRAINING_COMMIT = "047f04ee3bce0bd255f6e46731f36201e96e15e9"
REPO = Path("/kaggle/working/mvdec_fpmax")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", TRAINING_COMMIT], cwd=REPO, check=True)
resolved = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert resolved == TRAINING_COMMIT, (resolved, TRAINING_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
sys.path.insert(0, str(REPO / "src"))
from sklearn import __version__ as sklearn_version
assert sklearn_version == "1.6.1", sklearn_version
print("scikit-learn:", sklearn_version)
subprocess.run([
    sys.executable, str(REPO / "scripts/fetch_dekm_release.py"),
    "--datasets", "REUTERS", "20NEWS", "RCV1",
], cwd=REPO, check=True)

from sklearn.datasets import fetch_20newsgroups, fetch_rcv1
fetch_20newsgroups(subset="all", download_if_missing=True)
fetch_rcv1(download_if_missing=True)
print("Canonical public sources and exact training commit are ready.")


In [ ]:
import json
import os
from pipeline.public_reproduction_matrix import run_public_matrix

PROTOCOL = "mvdec_2025_aligned_balanced_alignment_005_full_batch_v1"
OUTPUT_ROOT = Path("/kaggle/working/mvdec-aligned-balanced-alignment005-seed42-output")
ARCHIVE_ROOT = Path("/kaggle/working/mvdec-aligned-balanced-alignment005-seed42-archives")
DATASET_ROOT = REPO / "external_repos/DEKM/datasets"


def run_seed42(dataset):
    os.environ["PYTHONHASHSEED"] = "42"
    list(run_public_matrix(
        datasets=[dataset],
        protocols=[PROTOCOL],
        seeds=[42],
        dataset_root=DATASET_ROOT,
        output_root=OUTPUT_ROOT,
        archive_root=ARCHIVE_ROOT,
        progress_interval=100,
        heartbeat_seconds=300,
        resume=True,
    ))


def load_only_manifest(dataset):
    dataset_dir = dataset.lower()
    run_dirs = sorted((OUTPUT_ROOT / dataset_dir / PROTOCOL).glob("*/seed_42"))
    assert len(run_dirs) == 1, run_dirs
    manifest = json.loads((run_dirs[0] / "manifest.json").read_text())
    assert manifest["status"] == "complete"
    assert manifest["training_commit"] == TRAINING_COMMIT
    assert manifest["runtime"]["scikit_learn_version"] == "1.6.1"
    return manifest


# Run inexpensive public datasets first; each failure stops the notebook.
run_seed42("REUTERS")
reuters_manifest = load_only_manifest("REUTERS")
run_seed42("RCV1")
rcv1_manifest = load_only_manifest("RCV1")

# Run canonical sklearn 20NEWS last, as requested.
run_seed42("20NEWS")
news_manifest = load_only_manifest("20NEWS")
assert news_manifest["source_id"] == "sklearn_fetch_20newsgroups_all_tfidf2000_18846"
assert news_manifest["n_samples"] == 18_846


In [ ]:
import pandas as pd

runs_path = ARCHIVE_ROOT / "public_mvdec_matrix_runs.csv"
aggregate_path = ARCHIVE_ROOT / "public_mvdec_matrix_aggregate.csv"
runs = pd.read_csv(runs_path)
expected = {(dataset, 42) for dataset in ("REUTERS", "RCV1", "20NEWS")}
actual = set(zip(runs["dataset"], runs["seed"], strict=True))
assert actual == expected, (actual, expected)
assert runs["protocol_id"].eq(PROTOCOL).all()
assert runs["training_commit"].eq(TRAINING_COMMIT).all()
assert runs.loc[runs["dataset"] == "20NEWS", "source_id"].eq(
    "sklearn_fetch_20newsgroups_all_tfidf2000_18846"
).all()
aggregate = pd.read_csv(aggregate_path)
assert aggregate["completed_runs"].eq(1).all()
(ARCHIVE_ROOT / "failures.json").write_text("[]\n", encoding="utf-8")
display(runs[[
    "dataset", "protocol_id", "seed", "source_id",
    "view1_acc", "view1_nmi", "view2_acc", "view2_nmi",
    "acc", "nmi", "runtime_seconds",
]])
display(aggregate)


In [ ]:
import hashlib
import shutil

assert (ARCHIVE_ROOT / "failures.json").read_text(encoding="utf-8") == "[]\n"
zip_path = shutil.make_archive(
    "/kaggle/working/mvdec-aligned-balanced-alignment005-three-datasets-seed42",
    "zip",
    ARCHIVE_ROOT,
)
digest = hashlib.sha256(Path(zip_path).read_bytes()).hexdigest()
Path(f"{zip_path}.sha256").write_text(f"{digest}  {Path(zip_path).name}\n", encoding="utf-8")
print("ZIP:", zip_path)
print("SHA-256:", digest)
